In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.orders"
)

customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

addresses_df = (
    spark.table("customer360_dev.silver.addresses")
    .select("address_id")
)

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "order_timestamp_cast",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "gross_amount_cast",
        F.col("gross_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "discount_amount_cast",
        F.col("discount_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "tax_amount_cast",
        F.col("tax_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "shipping_amount_cast",
        F.col("shipping_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "net_amount_cast",
        F.col("net_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "order_status_clean",
        F.upper(F.trim("order_status"))
    )
    .withColumn(
        "currency_clean",
        F.upper(F.trim("currency_code"))
    )
)

In [0]:
validated_df = (
    typed_df
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

Add shipping-address referential check

In [0]:
shipping_addresses_df = (
    addresses_df
    .withColumnRenamed(
        "address_id",
        "shipping_address_id"
    )
    .withColumn(
        "_shipping_address_exists",
        F.lit(True)
    )
)

validated_df = (
    validated_df
    .join(
        shipping_addresses_df,
        on="shipping_address_id",
        how="left"
    )
)

Add billing-address referential check

In [0]:
billing_addresses_df = (
    addresses_df
    .withColumnRenamed(
        "address_id",
        "billing_address_id"
    )
    .withColumn(
        "_billing_address_exists",
        F.lit(True)
    )
)

validated_df = (
    validated_df
    .join(
        billing_addresses_df,
        on="billing_address_id",
        how="left"
    )
)

Define core data-quality rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_shipping_address_not_found",
        F.col("shipping_address_id").isNotNull()
        & F.col("_shipping_address_exists").isNull()
    )
    .withColumn(
        "dq_billing_address_not_found",
        F.col("billing_address_id").isNotNull()
        & F.col("_billing_address_exists").isNull()
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("order_status_clean").isin(
            "CREATED",
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        )
    )
)

Validate numeric amounts

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_invalid_gross_amount",
        F.col("gross_amount_cast").isNull()
        | (F.col("gross_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_discount_amount",
        F.col("discount_amount_cast").isNull()
        | (F.col("discount_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_tax_amount",
        F.col("tax_amount_cast").isNull()
        | (F.col("tax_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_shipping_amount",
        F.col("shipping_amount_cast").isNull()
        | (F.col("shipping_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_net_amount",
        F.col("net_amount_cast").isNull()
        | (F.col("net_amount_cast") < 0)
    )
)

Add order financial reconciliation

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "calculated_net_amount",
        (
            F.col("gross_amount_cast")
            - F.col("discount_amount_cast")
            + F.col("tax_amount_cast")
            + F.col("shipping_amount_cast")
        ).cast("decimal(14,2)")
    )
    .withColumn(
        "dq_amount_mismatch",
        F.abs(
            F.col("calculated_net_amount")
            - F.col("net_amount_cast")
        ) > F.lit(0.01)
    )
)

Now compare it to source net_amount

In [0]:
validated_df = validated_df.withColumn(
    "dq_amount_mismatch",
    (
        F.abs(
            F.col("calculated_net_amount")
            - F.col("net_amount_cast")
        ) > F.lit(0.01)
    )
)

Validate timestamp and currency

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_invalid_order_timestamp",
        F.col("order_timestamp_cast").isNull()
    )
    .withColumn(
        "dq_invalid_currency",
        ~F.col("currency_clean").isin(
            "INR",
            "USD"
        )
    )
)

Add business warnings

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_warning_zero_value_delivered_order",
        (
            (F.col("order_status_clean") == "DELIVERED")
            & (F.col("net_amount_cast") == 0)
        )
    )
    .withColumn(
        "dq_warning_discount_exceeds_gross",
        (
            F.col("discount_amount_cast")
            > F.col("gross_amount_cast")
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_order_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_shipping_address_not_found")
    | F.col("dq_billing_address_not_found")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_gross_amount")
    | F.col("dq_invalid_discount_amount")
    | F.col("dq_invalid_tax_amount")
    | F.col("dq_invalid_shipping_amount")
    | F.col("dq_invalid_net_amount")
    | F.col("dq_amount_mismatch")
    | F.col("dq_invalid_order_timestamp")
    | F.col("dq_invalid_currency")
)

Split valid and rejected rows

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

Deduplicate Orders deterministically

In [0]:
order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )
    .filter(F.col("_row_number") == 1)
)

Build final Silver Orders structure

In [0]:
silver_df = deduped_df.select(
    "order_id",
    "customer_id",

    F.col("order_timestamp_cast").alias(
        "order_timestamp"
    ),

    F.col("order_status_clean").alias(
        "order_status"
    ),

    F.col("currency_clean").alias(
        "currency"
    ),

    F.col("gross_amount_cast").alias(
        "gross_amount"
    ),

    F.col("discount_amount_cast").alias(
        "discount_amount"
    ),

    F.col("tax_amount_cast").alias(
        "tax_amount"
    ),

    F.col("shipping_amount_cast").alias(
        "shipping_amount"
    ),

    F.col("net_amount_cast").alias(
        "net_amount"
    ),

    "shipping_address_id",
    "billing_address_id",
    "promotion_code",

    "dq_warning_zero_value_delivered_order",
    "dq_warning_discount_exceeds_gross",

    F.col("created_at_cast").alias(
        "created_at"
    ),

    F.col("updated_at_cast").alias(
        "updated_at"
    ),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.orders"
    )
)

Build Quarantine Records

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("dq_missing_order_id"),
                F.lit("MISSING_ORDER_ID")
            ),

            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),

            F.when(
                F.col("dq_customer_not_found"),
                F.lit("CUSTOMER_NOT_FOUND")
            ),

            F.when(
                F.col("dq_shipping_address_not_found"),
                F.lit("SHIPPING_ADDRESS_NOT_FOUND")
            ),

            F.when(
                F.col("dq_billing_address_not_found"),
                F.lit("BILLING_ADDRESS_NOT_FOUND")
            ),

            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_ORDER_STATUS")
            ),

            F.when(
                F.col("dq_invalid_gross_amount"),
                F.lit("INVALID_GROSS_AMOUNT")
            ),

            F.when(
                F.col("dq_invalid_discount_amount"),
                F.lit("INVALID_DISCOUNT_AMOUNT")
            ),

            F.when(
                F.col("dq_invalid_tax_amount"),
                F.lit("INVALID_TAX_AMOUNT")
            ),

            F.when(
                F.col("dq_invalid_shipping_amount"),
                F.lit("INVALID_SHIPPING_AMOUNT")
            ),

            F.when(
                F.col("dq_invalid_net_amount"),
                F.lit("INVALID_NET_AMOUNT")
            ),

            F.when(
                F.col("dq_amount_mismatch"),
                F.lit("ORDER_AMOUNT_MISMATCH")
            ),

            F.when(
                F.col("dq_invalid_order_timestamp"),
                F.lit("INVALID_ORDER_TIMESTAMP")
            ),

            F.when(
                F.col("dq_invalid_currency"),
                F.lit("INVALID_CURRENCY")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.order_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.order_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.order_rejects"
    )
)

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.orders"
).count()

silver_count = spark.table(
    "customer360_dev.silver.orders"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.order_rejects"
).count()

print(f"Bronze Orders : {bronze_count:,}")
print(f"Silver Orders : {silver_count:,}")
print(f"Rejected      : {reject_count:,}")

Bronze Orders : 5,000,126
Silver Orders : 4,740,000
Rejected      : 260,126
